## 5️⃣ Excel Шаблоны (Экспорт/Импорт данных)

Работа с Excel шаблонами для загрузки и экспорта данных.

**Доступные шаблоны:**
-  - полный шаблон для всех моделей (Standard, Custom, Lite)
-  - упрощенный шаблон только для Standard и Lite моделей

**Подробнее:** 


In [ ]:
# Импорты и настройка
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import json
import subprocess
import os
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, HBox, VBox
from engine.database.data_mart import get_data_mart

# Настройка пути
ROOT = Path('../..').resolve()
print(f"📁 ROOT: {ROOT}")

# Список компаний
COMPANIES = [d.name for d in (ROOT / 'companies').iterdir() if d.is_dir() and not d.name.startswith('.')]
print(f"📊 Доступные компании: {COMPANIES}")


def load_company_csv(relative_path: str) -> pd.DataFrame:
    csv_path = ROOT / relative_path
    target_output = globals().get('legacy_output')
    if csv_path.exists():
        if target_output is not None:
            with target_output:
                print(f"⚠️ Legacy CSV: {csv_path.relative_to(ROOT)}")
        else:
            print(f"⚠️ Legacy CSV: {csv_path.relative_to(ROOT)}")
        try:
            return pd.read_csv(csv_path)
        except Exception as exc:
            if target_output is not None:
                with target_output:
                    print(f"⚠️ Ошибка чтения {relative_path}: {exc}")
            else:
                print(f"⚠️ Ошибка чтения {relative_path}: {exc}")
            return pd.DataFrame()
    return pd.DataFrame()


In [ ]:
# ===== ВИДЖЕТЫ ДЛЯ ВЫБОРА КОМПАНИИ =====
company_dropdown = widgets.Dropdown(
    options=COMPANIES,
    value='rusal' if 'rusal' in COMPANIES else COMPANIES[0] if COMPANIES else None,
    description='Компания:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

load_button = widgets.Button(
    description='📂 Загрузить конфигурацию',
    button_style='info',
    icon='folder-open'
)

save_button = widgets.Button(
    description='💾 Сохранить конфигурацию',
    button_style='success',
    icon='save'
)

reset_button = widgets.Button(
    description='🔄 Сбросить к шаблону',
    button_style='warning',
    icon='refresh'
)

status_output = widgets.Output()
mart_summary_output = widgets.Output()
legacy_output = widgets.Output()

display(HBox([company_dropdown, load_button, save_button, reset_button]))
display(status_output)
display(mart_summary_output)
display(legacy_output)


In [ ]:
# Глобальная переменная для текущей конфигурацииcurrent_config = {}current_company = Nonedef summarize_data_mart(company_name: str):    """Отображает информацию о витрине и предупреждает о legacy CSV."""    legacy_output.clear_output()    with mart_summary_output:        mart_summary_output.clear_output()        mart = None        try:            mart = get_data_mart(ROOT, company_name)            versions = mart.get_existing_versions()            if versions:                display(Markdown(f"**📦 Версия модели:** {versions[0]['version']}"))            else:                display(Markdown("⚠️ Версии модели не найдены"))            history_map = {                "IS": mart.get_history_income_statement(canonical=True),                "BS": mart.get_history_balance_sheet(canonical=True),                "CF": mart.get_history_cash_flow(canonical=True),            }            display(Markdown("### История (каноническая форма)"))            for stmt, df in history_map.items():                if df.empty:                    display(Markdown(f"- {stmt}: ⚠️ нет данных"))                    load_company_csv(f"companies/{company_name}/history/{stmt.lower()}_history_{company_name}.csv")                else:                    years = [c for c in df.columns if str(c).isdigit()]                    display(Markdown(f"- {stmt}: ✅ {len(years)} лет ({years[0]}-{years[-1]})"))            if versions:                legacy_map = {                    "IS": f"companies/{company_name}/outputs/model/3statement_IS.csv",                    "BS": f"companies/{company_name}/outputs/model/3statement_BS.csv",                    "CF": f"companies/{company_name}/outputs/model/3statement_CF.csv",                    "DEBT": f"companies/{company_name}/outputs/model/debt_waterfall.csv",                }                for stmt, path in legacy_map.items():                    df = mart.get_model_results(versions[0]['version'], stmt, canonical=(stmt in {"IS", "BS", "CF"}))                    status = "⚠️ нет данных" if df.empty else f"✅ {df.shape[0]} метрик"                    display(Markdown(f"- {stmt} forecast: {status}"))                    if df.empty:                        load_company_csv(path)        except Exception as exc:            display(Markdown(f"⚠️ Не удалось подключиться к Data Mart: {exc}"))        finally:            if mart is not None:                mart.close()def load_config(company_name):    """Загрузка конфигурации из project.yaml"""    global current_config, current_company        croot = ROOT / "companies" / company_name    proj_yaml_path = croot / "configs" / "project.yaml"        if proj_yaml_path.exists():        with open(proj_yaml_path, 'r', encoding='utf-8') as f:            current_config = yaml.safe_load(f) or {}        current_company = company_name                with status_output:            status_output.clear_output()            print(f"✅ Конфигурация загружена для компании: {company_name}")                summarize_data_mart(company_name)        return current_config    else:        # Загружаем шаблон        template_path = ROOT / "templates" / "project_template.yaml"        if template_path.exists():            with open(template_path, 'r', encoding='utf-8') as f:                current_config = yaml.safe_load(f) or {}                        with status_output:                status_output.clear_output()                print(f"📋 Загружен шаблон для новой компании: {company_name}")                        summarize_data_mart(company_name)            return current_config        else:            with status_output:                status_output.clear_output()                print(f"❌ Ни конфигурация, ни шаблон не найдены")            summarize_data_mart(company_name)            return {}def save_config():    """Сохранение конфигурации в project.yaml"""    global current_config, current_company        if not current_company:        with status_output:            status_output.clear_output()            print("❌ Сначала загрузите конфигурацию")        return        croot = ROOT / "companies" / current_company    proj_yaml_path = croot / "configs" / "project.yaml"    proj_yaml_path.parent.mkdir(parents=True, exist_ok=True)        # Валидация перед сохранением    validation_errors = validate_config(current_config)    if validation_errors:        with status_output:            status_output.clear_output()            print("❌ Ошибки валидации:")            for error in validation_errors:                print(f"  - {error}")        return False        try:        with open(proj_yaml_path, 'w', encoding='utf-8') as f:            yaml.safe_dump(current_config, f, allow_unicode=True, sort_keys=False, default_flow_style=False, indent=2)                with status_output:            status_output.clear_output()            print(f"✅ Конфигурация сохранена: {proj_yaml_path}")                # Обновить виджеты с новыми значениями        update_widgets_from_config()                return True    except Exception as e:        with status_output:            status_output.clear_output()            print(f"❌ Ошибка при сохранении: {e}")        return Falsedef reset_to_template():    """Сброс к шаблону"""    global current_config        template_path = ROOT / "templates" / "project_template.yaml"    if template_path.exists():        with open(template_path, 'r', encoding='utf-8') as f:            current_config = yaml.safe_load(f) or {}                update_widgets_from_config()                with status_output:            status_output.clear_output()            print("🔄 Конфигурация сброшена к шаблону")def validate_config(config):    """Валидация конфигурации"""    errors = []        # Проверка обязательных полей    if "model" not in config:        errors.append("Отсутствует раздел 'model'")        if "macro_forecast" not in config:        errors.append("Отсутствует раздел 'macro_forecast'")        # Проверка значений    if "model" in config and "standard" in config["model"]:        debt_config = config["model"]["standard"].get("debt", {})        if debt_config.get("rc", {}).get("enable", False):            if debt_config["rc"].get("limit", 0) <= 0:                errors.append("RC limit должен быть > 0 если RC включен")        return errorsdef update_widgets_from_config():    """Обновление виджетов из конфигурации"""    global current_config        # Обновляем виджеты только если они уже созданы    try:        # Revenue        revenue_config = current_config.get("model", {}).get("standard", {}).get("revenue", {})        if 'revenue_driver' in globals():            revenue_driver.value = revenue_config.get("driver", "steel_price_hrc")            revenue_elasticity.value = revenue_config.get("elasticity", 1.0)            use_elastic_net.value = revenue_config.get("use_elastic_net", True)            fallback_growth.value = revenue_config.get("fallback_growth", "ewa")                # Debt        debt_config = current_config.get("model", {}).get("standard", {}).get("debt", {})        rc_config = debt_config.get("rc", {})        refinance_config = debt_config.get("refinancing", {})                if 'rc_enable' in globals():            rc_enable.value = rc_config.get("enable", True)            rc_limit.value = rc_config.get("limit", 2000.0)            rc_min_cash.value = rc_config.get("min_cash", 500.0)            rc_rate_spread.value = rc_config.get("rate_spread", 0.03)            refinance_enable.value = refinance_config.get("enable", True)            refinance_tenor.value = refinance_config.get("default_tenor_years", 5)            bonds_refinance_tenor.value = refinance_config.get("bonds_refinance_tenor_years", 3)            refinance_spread.value = refinance_config.get("refinance_rate_spread", 0.02)                # Train/Test        train_test_config = current_config.get("model", {}).get("standard", {}).get("train_test", {})        if 'train_end_year' in globals():            train_end_year.value = train_test_config.get("train_end_year", 2022)                # Periods        periods_config = current_config.get("model", {}).get("standard", {}).get("periods", {})        if 'history_start_year' in globals():            history_start_year.value = periods_config.get("history_start_year", 2010)            history_end_year.value = periods_config.get("history_end_year", 2024)            forecast_start_year.value = periods_config.get("forecast_start_year", 2025)            forecast_end_year.value = periods_config.get("forecast_end_year", 2029)            forecast_years.value = periods_config.get("forecast_years", 5)                # WC        wc_config = current_config.get("model", {}).get("standard", {}).get("wc", {})        if 'wc_method' in globals():            wc_method.value = wc_config.get("method", "days")            dso_days.value = wc_config.get("dso_days", 45)            dio_days.value = wc_config.get("dio_days", 60)            dpo_days.value = wc_config.get("dpo_days", 50)                # COGS        cogs_config = current_config.get("model", {}).get("standard", {}).get("cogs", {})        if 'use_ppi_uplift' in globals():            use_ppi_uplift.value = cogs_config.get("use_ppi_uplift", False)            ppi_series.value = cogs_config.get("ppi_series", "ppi_us")            ppi_beta.value = cogs_config.get("ppi_beta", 1.0)            cogs_min.value = cogs_config.get("clamp", {}).get("min", 0.55)            cogs_max.value = cogs_config.get("clamp", {}).get("max", 0.98)                # CapEx        capex_config = current_config.get("model", {}).get("standard", {}).get("capex_policy", {})        if 'capex_method' in globals():            capex_method.value = capex_config.get("method", "ratio_to_revenue")            capex_ratio.value = capex_config.get("ratio_default", 0.08)            maint_as_dep_ratio.value = capex_config.get("maint_as_dep_ratio", 1.0)                # PP&E        ppe_config = current_config.get("model", {}).get("standard", {}).get("ppe", {})        if 'ppe_mode' in globals():            ppe_mode.value = ppe_config.get("mode", "full")            ppe_net_pct.value = ppe_config.get("net_pct_revenue", 0.75)            ppe_useful_life.value = ppe_config.get("useful_life_years", 12)                # Leases        leases_config = current_config.get("model", {}).get("standard", {}).get("leases", {})        if 'leases_enabled' in globals():            leases_enabled.value = leases_config.get("enabled", False)            lease_discount_rate.value = leases_config.get("default_discount_rate", 0.05)                # Taxes        taxes_config = current_config.get("model", {}).get("standard", {}).get("taxes", {})        margins_config = current_config.get("model", {}).get("standard", {}).get("margins", {})        if 'tax_mode' in globals():            tax_mode.value = taxes_config.get("mode", "full")            tax_rate_stat.value = margins_config.get("tax_rate_statutory", 0.21)            nol_balance.value = taxes_config.get("nol_opening_balance", 0.0)                # Intangibles        intang_config = current_config.get("model", {}).get("standard", {}).get("intangibles", {})        if 'intang_mode' in globals():            intang_mode.value = intang_config.get("mode", "full")            intang_pct.value = intang_config.get("intang_pct_revenue", 0.0)            intang_amort_rate.value = intang_config.get("amortization_pct_of_balance", 10.0)                # Equity        equity_config = current_config.get("model", {}).get("standard", {}).get("equity", {})        if 'equity_mode' in globals():            equity_mode.value = equity_config.get("mode", "full")            dividend_payout.value = equity_config.get("dividend_payout_ratio", 0.0)                # Features        features_config = current_config.get("features", {})        if 'use_ppe_corkscrew' in globals():            use_ppe_corkscrew.value = features_config.get("use_ppe_corkscrew", True)            use_wc_days.value = features_config.get("use_wc_days", True)            use_debt_rc.value = features_config.get("use_debt_rc", True)            use_intangibles_corkscrew.value = features_config.get("use_intangibles_corkscrew", True)                # YAML Editor        # YAML Editor        if 'yaml_editor' in globals():            yaml_editor.value = yaml.dump(current_config, allow_unicode=True, sort_keys=False, default_flow_style=False, indent=2)    except Exception as e:        pass  # Виджеты еще не созданы# Привязка обработчиковload_button.on_click(lambda b: load_config(company_dropdown.value))save_button.on_click(lambda b: save_config())reset_button.on_click(lambda b: reset_to_template())# Автозагрузка при изменении компанииdef on_company_change(change):    if change['type'] == 'change' and change['name'] == 'value':        load_config(change['new'])company_dropdown.observe(on_company_change)# Первоначальная загрузкаload_config(company_dropdown.value)

## 1️⃣ Макро-факторы и файлы


In [ ]:
# Виджеты для макро-факторов
macro_section = widgets.Accordion([
    widgets.VBox([
        widgets.HTML("<b>Список факторов (через запятую):</b>"),
        widgets.Textarea(
            value=', '.join(current_config.get("macro_forecast", {}).get("factors", [])),
            description='Факторы:',
            layout=Layout(width='600px', height='80px')
        ),
        widgets.HTML("<i>Пример: gdp_us, industrial_production_us, steel_price_hrc, cpi_us</i>"),
    ])
], selected_index=None)
macro_section.set_title(0, '📊 Макро-факторы')

display(macro_section)


## 2️⃣ Revenue Modeling (Моделирование выручки)


## ⚙️ Настройка model.mode: Standard vs Custom

Модель поддерживает два режима работы:

- **standard**: упрощенная методология с базовыми расчетами
- **custom**: продвинутая методология с полными schedules

### Пример конфигурации:

```yaml
model:
  mode: custom  # standard | custom
  features:
    use_ppe_corkscrew: true      # Custom only
    use_tax_schedule: true        # Custom only
    use_lease_schedule: true      # Custom only
    use_wc_days: true             # Available for both
    use_debt_rc: true             # Available for both (simplified for Standard)
```

### Особенности:
- **Standard mode** автоматически отключает продвинутые features
- **Custom mode** поддерживает оба варианта (простой и corkscrew)
- **Макродвижок** доступен обоим режимам для Revenue
- Все **параметры рассчитываются в движке** из истории

**Документация:** `docs/MODEL_ENGINES_COMPARISON.md`


In [ ]:
# Параметры моделирования Revenue
revenue_config = current_config.get("model", {}).get("standard", {}).get("revenue", {})

revenue_driver = widgets.Dropdown(
    options=['steel_price_hrc', 'gdp_us', 'industrial_production_us', 'custom'],
    value=revenue_config.get("driver", "steel_price_hrc"),
    description='Основной драйвер:',
    style={'description_width': '200px'}
)

revenue_elasticity = widgets.FloatSlider(
    value=revenue_config.get("elasticity", 1.0),
    min=0.1,
    max=3.0,
    step=0.1,
    description='Elasticity:',
    style={'description_width': '200px'}
)

use_elastic_net = widgets.Checkbox(
    value=revenue_config.get("use_elastic_net", True),
    description='Использовать Elastic Net',
    style={'description_width': '200px'}
)

l1_ratio = widgets.FloatSlider(
    value=revenue_config.get("l1_ratio", 0.5),
    min=0.0,
    max=1.0,
    step=0.1,
    description='L1 Ratio (Elastic Net):',
    style={'description_width': '200px'}
)

fallback_growth = widgets.Dropdown(
    options=['ewa', 'constant', 'gdp'],
    value=revenue_config.get("fallback_growth", "ewa"),
    description='Fallback метод:',
    style={'description_width': '200px'}
)

revenue_section = widgets.VBox([
    widgets.HTML("<h4>📈 Параметры моделирования Revenue</h4>"),
    revenue_driver,
    revenue_elasticity,
    use_elastic_net,
    l1_ratio,
    fallback_growth,
    widgets.HTML("<i>Elasticity: чувствительность выручки к драйверу (1.0 = пропорциональная)</i>"),
    widgets.HTML("<i>Elastic Net: комбинация L1 (Lasso) и L2 (Ridge) регуляризации. L1 Ratio: 0.0 = Ridge (только L2), 1.0 = Lasso (только L1), 0.5 = Elastic Net (равная смесь)</i>")
])

display(revenue_section)


## 📦 Disposal Proceeds и Gain/Loss on Disposal

### Standard Mode:
- **Gain/Loss on Disposal**: fallback = 0
- **Disposal Proceeds**: fallback = 0 (в CFI)

### Custom Mode:
- **Gain/Loss on Disposal**: из PP&E schedule (если `use_ppe_corkscrew: true`)
- **Disposal Proceeds**: из PP&E schedule (в CFI через `cfi_other`)

Переменные автоматически инициализируются перед итерационным циклом.


## 3️⃣ Debt & Refinancing (Долг и рефинансирование)


In [ ]:
# Параметры долга
debt_config = current_config.get("model", {}).get("standard", {}).get("debt", {})
rc_config = debt_config.get("rc", {})

rc_enable = widgets.Checkbox(
    value=rc_config.get("enable", True),
    description='Включить Revolving Credit',
    style={'description_width': '250px'}
)

rc_limit = widgets.FloatText(
    value=rc_config.get("limit", 2000.0),
    description='RC Limit (млн):',
    style={'description_width': '250px'}
)

rc_min_cash = widgets.FloatText(
    value=rc_config.get("min_cash", 500.0),
    description='Min Cash (млн):',
    style={'description_width': '250px'}
)

rc_rate_spread = widgets.FloatSlider(
    value=rc_config.get("rate_spread", 0.03),
    min=0.0,
    max=0.10,
    step=0.001,
    description='Rate Spread:',
    style={'description_width': '250px'}
)

# Рефинансирование
refinance_config = debt_config.get("refinancing", {})

refinance_enable = widgets.Checkbox(
    value=refinance_config.get("enable", True),
    description='Включить рефинансирование',
    style={'description_width': '250px'}
)

refinance_tenor = widgets.IntText(
    value=refinance_config.get("default_tenor_years", 5),
    description='Tenor для займов (лет):',
    style={'description_width': '250px'}
)

bonds_refinance_tenor = widgets.IntText(
    value=refinance_config.get("bonds_refinance_tenor_years", 3),
    description='Tenor для бондов (лет):',
    style={'description_width': '250px'}
)

refinance_spread = widgets.FloatSlider(
    value=refinance_config.get("refinance_rate_spread", 0.02),
    min=0.0,
    max=0.10,
    step=0.001,
    description='Refinance Spread:',
    style={'description_width': '250px'}
)

debt_section = widgets.VBox([
    widgets.HTML("<h4>💳 Параметры Debt & Refinancing</h4>"),
    widgets.HTML("<b>Revolving Credit:</b>"),
    rc_enable,
    rc_limit,
    rc_min_cash,
    rc_rate_spread,
    widgets.HTML("<br><b>Рефинансирование:</b>"),
    refinance_enable,
    refinance_tenor,
    bonds_refinance_tenor,
    refinance_spread,
])

display(debt_section)


## 4️⃣ Train/Test Split


ст

## 6️⃣ Working Capital (WC)

In [ ]:
# Working Capital настройкиwc_config = current_config.get("model", {}).get("standard", {}).get("wc", {})wc_method = widgets.Dropdown(    options=['days', 'ratio'],    value=wc_config.get("method", "days"),    description='WC Method:',    style={'description_width': '200px'})dso_days = widgets.FloatText(    value=wc_config.get("dso_days", 45),    description='DSO Days:',    style={'description_width': '200px'})dio_days = widgets.FloatText(    value=wc_config.get("dio_days", 60),    description='DIO Days:',    style={'description_width': '200px'})dpo_days = widgets.FloatText(    value=wc_config.get("dpo_days", 50),    description='DPO Days:',    style={'description_width': '200px'})wc_section = widgets.VBox([    widgets.HTML("<h4>💼 Working Capital</h4>"),    wc_method,    dso_days,    dio_days,    dpo_days,    widgets.HTML("<i>Метод расчета WC: days (в днях) или ratio (в % от Revenue)</i>")])display(wc_section)

## 7️⃣ COGS (Себестоимость)

In [ ]:
# COGS настройкиcogs_config = current_config.get("model", {}).get("standard", {}).get("cogs", {})use_ppi_uplift = widgets.Checkbox(    value=cogs_config.get("use_ppi_uplift", False),    description='Использовать PPI индексацию',    style={'description_width': '250px'})ppi_series = widgets.Text(    value=cogs_config.get("ppi_series", "ppi_us"),    description='PPI Series:',    style={'description_width': '250px'})ppi_beta = widgets.FloatText(    value=cogs_config.get("ppi_beta", 1.0),    description='PPI Beta:',    style={'description_width': '250px'})cogs_min = widgets.FloatText(    value=cogs_config.get("clamp", {}).get("min", 0.55),    description='COGS Min (%):',    style={'description_width': '250px'})cogs_max = widgets.FloatText(    value=cogs_config.get("clamp", {}).get("max", 0.98),    description='COGS Max (%):',    style={'description_width': '250px'})cogs_section = widgets.VBox([    widgets.HTML("<h4>💰 COGS (Себестоимость)</h4>"),    use_ppi_uplift,    ppi_series,    ppi_beta,    widgets.HTML("<b>Ограничения:</b>"),    cogs_min,    cogs_max,    widgets.HTML("<i>PPI Beta: 1.0 = полная индексация, 'history' = авторасчет</i>")])display(cogs_section)

## 8️⃣ CapEx Policy

In [ ]:
# CapEx Policy настройкиcapex_config = current_config.get("model", {}).get("standard", {}).get("capex_policy", {})capex_method = widgets.Dropdown(    options=['ratio_to_revenue', 'external_file', 'fixed'],    value=capex_config.get("method", "ratio_to_revenue"),    description='CapEx Method:',    style={'description_width': '250px'})capex_ratio = widgets.FloatText(    value=capex_config.get("ratio_default", 0.08),    description='CapEx Ratio (%):',    style={'description_width': '250px'})maint_as_dep_ratio = widgets.FloatText(    value=capex_config.get("maint_as_dep_ratio", 1.0),    description='Maint/Dep Ratio:',    style={'description_width': '250px'})capex_section = widgets.VBox([    widgets.HTML("<h4>🏗️ CapEx Policy</h4>"),    capex_method,    capex_ratio,    maint_as_dep_ratio,    widgets.HTML("<i>CapEx Ratio: % от Revenue по умолчанию</i>")])display(capex_section)

## 9️⃣ PP&E (Основные средства)

In [ ]:
# PP&E настройкиppe_config = current_config.get("model", {}).get("standard", {}).get("ppe", {})ppe_mode = widgets.Dropdown(    options=['full', 'lite', 'standard'],    value=ppe_config.get("mode", "full"),    description='PP&E Mode:',    style={'description_width': '200px'})ppe_net_pct = widgets.FloatText(    value=ppe_config.get("net_pct_revenue", 0.75),    description='Net PP&E % Revenue:',    style={'description_width': '200px'})ppe_useful_life = widgets.FloatText(    value=ppe_config.get("useful_life_years", 12),    description='Useful Life (years):',    style={'description_width': '200px'})ppe_section = widgets.VBox([    widgets.HTML("<h4>🏭 PP&E (Основные средства)</h4>"),    ppe_mode,    ppe_net_pct,    ppe_useful_life,    widgets.HTML("<i>Mode: full (corkscrew), lite (Net only), standard (Net = % Revenue)</i>")])display(ppe_section)

## 🔟 Leases (Лизинг)

In [ ]:
# Leases настройкиleases_config = current_config.get("model", {}).get("standard", {}).get("leases", {})leases_enabled = widgets.Checkbox(    value=leases_config.get("enabled", False),    description='Включить лизинг',    style={'description_width': '250px'})lease_discount_rate = widgets.FloatText(    value=leases_config.get("default_discount_rate", 0.05),    description='Discount Rate:',    style={'description_width': '250px'})leases_section = widgets.VBox([    widgets.HTML("<h4>📋 Leases (Лизинг)</h4>"),    leases_enabled,    lease_discount_rate,    widgets.HTML("<i>IFRS 16 / ASC 842 лизинг</i>")])display(leases_section)

## 1️⃣1️⃣ Taxes (Налоги)

In [ ]:
# Taxes настройкиtaxes_config = current_config.get("model", {}).get("standard", {}).get("taxes", {})margins_config = current_config.get("model", {}).get("standard", {}).get("margins", {})tax_mode = widgets.Dropdown(    options=['full', 'lite', 'standard'],    value=taxes_config.get("mode", "full"),    description='Tax Mode:',    style={'description_width': '200px'})tax_rate_stat = widgets.FloatText(    value=margins_config.get("tax_rate_statutory", 0.21),    description='Statutory Rate:',    style={'description_width': '200px'})nol_balance = widgets.FloatText(    value=taxes_config.get("nol_opening_balance", 0.0),    description='NOL Opening Balance:',    style={'description_width': '200px'})taxes_section = widgets.VBox([    widgets.HTML("<h4>💸 Taxes (Налоги)</h4>"),    tax_mode,    tax_rate_stat,    nol_balance,    widgets.HTML("<i>Mode: full (NOL+DTL), lite (stat rate), standard (effective rate)</i>")])display(taxes_section)

## 1️⃣2️⃣ Intangibles (Нематериальные активы)

In [ ]:
# Intangibles настройкиintang_config = current_config.get("model", {}).get("standard", {}).get("intangibles", {})intang_mode = widgets.Dropdown(    options=['full', 'lite', 'standard'],    value=intang_config.get("mode", "full"),    description='Intangibles Mode:',    style={'description_width': '250px'})intang_pct = widgets.FloatText(    value=intang_config.get("intang_pct_revenue", 0.0),    description='Intangibles % Revenue:',    style={'description_width': '250px'})intang_amort_rate = widgets.FloatText(    value=intang_config.get("amortization_pct_of_balance", 10.0),    description='Amortization Rate (%):',    style={'description_width': '250px'})intang_section = widgets.VBox([    widgets.HTML("<h4>📜 Intangibles (Нематериальные активы)</h4>"),    intang_mode,    intang_pct,    intang_amort_rate,    widgets.HTML("<i>Mode: full (corkscrew), lite (fixed), standard (% Revenue)</i>")])display(intang_section)

## 1️⃣3️⃣ Equity (Капитал)

In [ ]:
# Equity настройкиequity_config = current_config.get("model", {}).get("standard", {}).get("equity", {})equity_mode = widgets.Dropdown(    options=['full', 'lite', 'standard'],    value=equity_config.get("mode", "full"),    description='Equity Mode:',    style={'description_width': '200px'})dividend_payout = widgets.FloatText(    value=equity_config.get("dividend_payout_ratio", 0.0),    description='Dividend Payout Ratio:',    style={'description_width': '200px'})equity_section = widgets.VBox([    widgets.HTML("<h4>💎 Equity (Капитал)</h4>"),    equity_mode,    dividend_payout,    widgets.HTML("<i>Mode: full (dividends), lite (disabled), standard (dividends+issuance+buyback)</i>")])display(equity_section)

## 1️⃣4️⃣ Features (Флаги функций)

In [ ]:
# Features настройкиfeatures_config = current_config.get("features", {})use_ppe_corkscrew = widgets.Checkbox(    value=features_config.get("use_ppe_corkscrew", True),    description='PP&E Corkscrew',    style={'description_width': '200px'})use_wc_days = widgets.Checkbox(    value=features_config.get("use_wc_days", True),    description='WC Days',    style={'description_width': '200px'})use_debt_rc = widgets.Checkbox(    value=features_config.get("use_debt_rc", True),    description='Debt RC',    style={'description_width': '200px'})use_intangibles_corkscrew = widgets.Checkbox(    value=features_config.get("use_intangibles_corkscrew", True),    description='Intangibles Corkscrew',    style={'description_width': '200px'})features_section = widgets.VBox([    widgets.HTML("<h4>⚙️ Features (Флаги функций)</h4>"),    use_ppe_corkscrew,    use_wc_days,    use_debt_rc,    use_intangibles_corkscrew,    widgets.HTML("<i>Включение/выключение функций модели</i>")])display(features_section)

In [ ]:
# Виджеты для работы с Excel шаблонамиexcel_output = widgets.Output()# Определение путей к шаблонамTEMPLATE_UNIFIED = ROOT / "templates" / "excel_templates" / "template_UNIFIED_All_Data.xlsx"TEMPLATE_STANDARD = ROOT / "templates" / "excel_templates" / "template_STANDARD_LITE.xlsx"# Выбор типа шаблонаtemplate_type = widgets.Dropdown(    options=[        ('Полный (UNIFIED)', 'unified'),        ('Упрощенный (STANDARD/LITE)', 'standard')    ],    value='unified',    description='Тип шаблона:',    style={'description_width': '200px'},    layout=Layout(width='400px'))# Информация о шаблонахdef get_template_info(template_type_value):    """Получает информацию о выбранном шаблоне"""    if template_type_value == 'unified':        template_path = TEMPLATE_UNIFIED        description = "Полный шаблон для всех моделей (Standard, Custom, Lite)\nСодержит все метрики из БД"    else:        template_path = TEMPLATE_STANDARD        description = "Упрощенный шаблон только для Standard и Lite моделей\nСодержит только канонические метрики"        exists = template_path.exists()    size = template_path.stat().st_size / 1024 if exists else 0        return {        'path': template_path,        'exists': exists,        'size': size,        'description': description    }template_info_output = widgets.Output()def update_template_info(change=None):    """Обновляет информацию о шаблоне"""    with template_info_output:        template_info_output.clear_output()        info = get_template_info(template_type.value)                if info['exists']:            display(Markdown(f"**📄 Шаблон:** `{info['path'].name}`"))            display(Markdown(f"**📁 Путь:** `{info['path']}`"))            display(Markdown(f"**📊 Размер:** {info['size']:.1f} KB"))            display(Markdown(f"**ℹ️ Описание:** {info['description']}"))        else:            display(Markdown(f"**❌ Шаблон не найден:** `{info['path']}`"))            display(Markdown("**💡 Создайте шаблон с помощью скрипта:**"))            display(Markdown("```bash\npython tools/create_standard_template.py\n```"))template_type.observe(update_template_info, names='value')update_template_info()# Кнопки для работы с Excelexport_button = widgets.Button(    description='📤 Экспорт в Excel',    button_style='info',    icon='upload')import_button = widgets.Button(    description='📥 Импорт из Excel',    button_style='success',    icon='download')def export_to_excel(b):    """Экспорт данных из БД в Excel шаблон"""    if not current_company:        with excel_output:            excel_output.clear_output()            print("❌ Сначала выберите компанию")        return        info = get_template_info(template_type.value)    if not info['exists']:        with excel_output:            excel_output.clear_output()            print(f"❌ Шаблон не найден: {info['path']}")        return        # Определяем путь для экспорта    company_root = ROOT / "companies" / current_company    excel_output_path = company_root / "data" / "excel" / f"{current_company}_exported.xlsx"    excel_output_path.parent.mkdir(parents=True, exist_ok=True)        # Команда экспорта    export_cmd = [        "python3",        str(ROOT / "tools" / "export_data_to_excel_template.py"),        "--company", current_company,        "--output", str(excel_output_path),        "--template", str(info['path'])    ]        with excel_output:        excel_output.clear_output()        print(f"🚀 Экспорт данных в Excel...")        print(f"📄 Шаблон: {info['path'].name}")        print(f"📁 Выходной файл: {excel_output_path}")        print(f"\nКоманда:")        print(f"  {' '.join(export_cmd)}")        print(f"\n⚠️ Для реального экспорта раскомментируйте код ниже:")                # Раскомментируйте для реального экспорта:        # env = os.environ.copy()        # env['PYTHONPATH'] = str(ROOT) + ':' + env.get('PYTHONPATH', '')        # result = subprocess.run(export_cmd, cwd=ROOT, env=env, capture_output=True, text=True)        # if result.returncode == 0:        #     print("✅ Экспорт выполнен успешно!")        #     if excel_output_path.exists():        #         print(f"📄 Файл создан: {excel_output_path}")        #         print(f"   Размер: {excel_output_path.stat().st_size / 1024:.1f} KB")        # else:        #     print("❌ Ошибка при экспорте:")        #     print(result.stderr)def import_from_excel(b):    """Импорт данных из Excel в БД"""    if not current_company:        with excel_output:            excel_output.clear_output()            print("❌ Сначала выберите компанию")        return        # Определяем путь к Excel файлу    company_root = ROOT / "companies" / current_company    excel_input_path = company_root / "data" / "excel" / f"{current_company}_filled.xlsx"        if not excel_input_path.exists():        excel_input_path = company_root / "data" / "excel" / f"{current_company}_exported.xlsx"        if not excel_input_path.exists():        with excel_output:            excel_output.clear_output()            print(f"❌ Excel файл не найден")            print(f"   Ожидаемые пути:")            print(f"   - {company_root / 'data' / 'excel' / f'{current_company}_filled.xlsx'}")            print(f"   - {company_root / 'data' / 'excel' / f'{current_company}_exported.xlsx'}")        return        # Команда импорта    import_cmd = [        "python3",        str(ROOT / "tools" / "load_excel_to_data_mart.py"),        "--company", current_company,        "--file", str(excel_input_path)    ]        with excel_output:        excel_output.clear_output()        print(f"🚀 Импорт данных из Excel...")        print(f"📄 Файл: {excel_input_path}")        print(f"\nКоманда:")        print(f"  {' '.join(import_cmd)}")        print(f"\n⚠️ Для реального импорта раскомментируйте код ниже:")        print(f"⚠️ ВНИМАНИЕ: Это перезапишет данные в БД!")                # Раскомментируйте для реального импорта:        # env = os.environ.copy()        # env['PYTHONPATH'] = str(ROOT) + ':' + env.get('PYTHONPATH', '')        # result = subprocess.run(import_cmd, cwd=ROOT, env=env, capture_output=True, text=True)        # if result.returncode == 0:        #     print("✅ Импорт выполнен успешно!")        #     print("📊 Данные загружены в БД")        # else:        #     print("❌ Ошибка при импорте:")        #     print(result.stderr)export_button.on_click(export_to_excel)import_button.on_click(import_from_excel)excel_section = widgets.VBox([    widgets.HTML("<h4>📊 Работа с Excel шаблонами</h4>"),    template_type,    template_info_output,    widgets.HTML("<br>"),    widgets.HTML("<b>Действия:</b>"),Работа с Excel шаблонами для загрузки и экспорта данных.**Доступные шаблоны:**- `template_UNIFIED_All_Data.xlsx` - полный шаблон для всех моделей (Standard, Custom, Lite)- `template_STANDARD_LITE.xlsx` - упрощенный шаблон только для Standard и Lite моделей**Подробнее:** `docs/EXCEL_TEMPLATES_GUIDE.md`вия:</b>"),    HBox([export_button, import_button]),    excel_output])display(excel_section)

In [ ]:
# Train/Test Split
train_test_config = current_config.get("model", {}).get("standard", {}).get("train_test", {})

train_end_year = widgets.IntText(
    value=train_test_config.get("train_end_year", 2022),
    description='Train End Year:',
    style={'description_width': '200px'}
)

train_test_section = widgets.VBox([
    widgets.HTML("<h4>📊 Train/Test Split для валидации</h4>"),
    train_end_year,
    widgets.HTML("<i>Год разделения: данные до этого года используются для обучения, после - для тестирования</i>")
])

display(train_test_section)


In [ ]:
# Виджеты для работы с Excel шаблонами


## 5️⃣ Применение изменений и предпросмотр


In [ ]:
# Функция применения всех измененийdef apply_all_changes():    """Применяет все изменения из виджетов к current_config"""    global current_config        # Revenue    if "model" not in current_config:        current_config["model"] = {}    if "standard" not in current_config["model"]:        current_config["model"]["standard"] = {}    if "revenue" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["revenue"] = {}        current_config["model"]["standard"]["revenue"] = {        "driver": revenue_driver.value,        "elasticity": revenue_elasticity.value,        "use_elastic_net": use_elastic_net.value,        "fallback_growth": fallback_growth.value    }        # Debt    if "debt" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["debt"] = {}    if "rc" not in current_config["model"]["standard"]["debt"]:        current_config["model"]["standard"]["debt"]["rc"] = {}    if "refinancing" not in current_config["model"]["standard"]["debt"]:        current_config["model"]["standard"]["debt"]["refinancing"] = {}        current_config["model"]["standard"]["debt"]["rc"] = {        "enable": rc_enable.value,        "limit": rc_limit.value,        "min_cash": rc_min_cash.value,        "rate_spread": rc_rate_spread.value    }        current_config["model"]["standard"]["debt"]["refinancing"] = {        "enable": refinance_enable.value,        "default_tenor_years": refinance_tenor.value,        "bonds_refinance_tenor_years": bonds_refinance_tenor.value,        "refinance_rate_spread": refinance_spread.value    }        # Periods    if "periods" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["periods"] = {}    current_config["model"]["standard"]["periods"] = {        "history_start_year": history_start_year.value,        "history_end_year": history_end_year.value,        "forecast_start_year": forecast_start_year.value,        "forecast_end_year": forecast_end_year.value,        "forecast_years": forecast_years.value    }        # WC    if "wc" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["wc"] = {}    current_config["model"]["standard"]["wc"] = {        "method": wc_method.value,        "dso_days": dso_days.value,        "dio_days": dio_days.value,        "dpo_days": dpo_days.value    }        # COGS    if "cogs" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["cogs"] = {}    current_config["model"]["standard"]["cogs"] = {        "use_ppi_uplift": use_ppi_uplift.value,        "ppi_series": ppi_series.value,        "ppi_beta": ppi_beta.value,        "clamp": {            "min": cogs_min.value,            "max": cogs_max.value        }    }        # CapEx    if "capex_policy" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["capex_policy"] = {}    current_config["model"]["standard"]["capex_policy"] = {        "method": capex_method.value,        "ratio_default": capex_ratio.value,        "maint_as_dep_ratio": maint_as_dep_ratio.value    }        # PP&E    if "ppe" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["ppe"] = {}    current_config["model"]["standard"]["ppe"] = {        "mode": ppe_mode.value,        "net_pct_revenue": ppe_net_pct.value,        "useful_life_years": ppe_useful_life.value    }        # Leases    if "leases" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["leases"] = {}    current_config["model"]["standard"]["leases"] = {        "enabled": leases_enabled.value,        "default_discount_rate": lease_discount_rate.value    }        # Taxes    if "taxes" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["taxes"] = {}    current_config["model"]["standard"]["taxes"] = {        "mode": tax_mode.value,        "nol_opening_balance": nol_balance.value    }    if "margins" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["margins"] = {}    current_config["model"]["standard"]["margins"]["tax_rate_statutory"] = tax_rate_stat.value        # Intangibles    if "intangibles" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["intangibles"] = {}    current_config["model"]["standard"]["intangibles"] = {        "mode": intang_mode.value,        "intang_pct_revenue": intang_pct.value,        "amortization_pct_of_balance": intang_amort_rate.value    }        # Equity    if "equity" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["equity"] = {}    current_config["model"]["standard"]["equity"] = {        "mode": equity_mode.value,        "dividend_payout_ratio": dividend_payout.value    }        # Features    if "features" not in current_config:        current_config["features"] = {}    current_config["features"] = {        "use_ppe_corkscrew": use_ppe_corkscrew.value,        "use_wc_days": use_wc_days.value,        "use_debt_rc": use_debt_rc.value,        "use_intangibles_corkscrew": use_intangibles_corkscrew.value    }        # Train/Test    # Train/Test    if "train_test" not in current_config["model"]["standard"]:        current_config["model"]["standard"]["train_test"] = {}    current_config["model"]["standard"]["train_test"]["train_end_year"] = train_end_year.value        # Макро-факторы (уже обновляются автоматически через observer)    # Но обновим на всякий случай    if "macro_forecast" not in current_config:        current_config["macro_forecast"] = {}    factors_list = [f.strip() for f in macro_factors_text.value.split(',') if f.strip()]    current_config["macro_forecast"]["factors"] = factors_list        # Обновляем YAML редактор    if 'yaml_editor' in globals():        yaml_editor.value = yaml.dump(current_config, allow_unicode=True, sort_keys=False, default_flow_style=False, indent=2)        with status_output:        status_output.clear_output()        print("✅ Изменения применены к конфигурации")# Кнопки применения и предпросмотраapply_button = widgets.Button(    description='✏️ Применить изменения',    button_style='primary')preview_button = widgets.Button(    description='👁️ Предпросмотр YAML',    button_style='info')preview_output = widgets.Output()def on_apply(b):    apply_all_changes()def on_preview(b):    apply_all_changes()    with preview_output:        preview_output.clear_output()        display(Markdown("### 📄 Предпросмотр YAML (измененные разделы):"))        print(yaml.dump(current_config, allow_unicode=True, sort_keys=False, default_flow_style=False, indent=2))apply_button.on_click(on_apply)preview_button.on_click(on_preview)display(HBox([apply_button, preview_button]))display(preview_output)# Применяем текущие значения при загрузкеapply_all_changes()

## 6️⃣ Расширенные настройки

Для более детальной настройки можно редактировать YAML напрямую или использовать дополнительные виджеты ниже.


In [ ]:
# Прямое редактирование YAML (расширенный режим)
yaml_editor = widgets.Textarea(
    value=yaml.dump(current_config, allow_unicode=True, sort_keys=False, default_flow_style=False, indent=2),
    description='YAML Editor:',
    layout=Layout(width='100%', height='400px'),
    style={'description_width': '150px'}
)

def update_config_from_yaml(b):
    """Обновляет конфигурацию из YAML редактора"""
    global current_config
    try:
        current_config = yaml.safe_load(yaml_editor.value) or {}
        update_widgets_from_config()
        with status_output:
            status_output.clear_output()
            print("✅ YAML загружен и применен")
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка парсинга YAML: {e}")

load_yaml_button = widgets.Button(
    description='📥 Загрузить из YAML редактора',
    button_style='warning'
)
load_yaml_button.on_click(update_config_from_yaml)

advanced_section = widgets.VBox([
    widgets.HTML("<h4>🔧 Расширенный редактор YAML</h4>"),
    yaml_editor,
    load_yaml_button,
    widgets.HTML("<i>Внимание: изменения в виджетах не синхронизируются с редактором автоматически. Используйте 'Применить изменения' перед редактированием.</i>")
])

display(advanced_section)
